In [1]:
import pandas as pd

# Load CSV (Dataset)
repo_df = pd.read_csv('characteristics_repo.csv')
markdown_irt_df = pd.read_csv('characteristics_irts_markdown.csv')
yaml_irt_df = pd.read_csv('characteristics_irts_yaml.csv')

print(repo_df.head())
print(markdown_irt_df.head())
print(yaml_irt_df.head())

                          full_name  is_fork  has_issues           created_at  \
0        josch/cycles_johnson_meyer    False        True  2012-07-04 12:02:27   
1   hantsy/angularjs-cakephp-sample    False        True  2013-11-06 12:56:34   
2          metabase/metabase-deploy    False       False  2015-10-08 00:01:18   
3  sakshamsharma/http-over-protocol    False        True  2016-08-30 10:53:30   
4                  sradley/overflow    False        True  2020-10-27 06:30:05   

                   last_modified            pushed_at main_language  \
0  Sun, 01 Jan 2023 16:35:16 GMT  2018-07-05 03:54:37          Java   
1  Wed, 07 Dec 2022 18:29:56 GMT  2015-10-08 05:00:41           PHP   
2  Sat, 10 Dec 2022 12:07:00 GMT  2022-08-05 22:57:15         Shell   
3  Tue, 15 Nov 2022 20:41:40 GMT  2016-09-25 04:59:55           C++   
4  Thu, 01 Dec 2022 11:13:38 GMT  2022-12-18 03:42:19            Go   

   total_issues_count  open_issues_count  closed_issues_count  ...  \
0               

In [2]:
rows1, columns1 = repo_df.shape

# Print the number of rows and columns
print(f"The repo dataset has {rows1} rows and {columns1} columns.")

The repo dataset has 1084300 rows and 29 columns.


In [3]:
rows2, columns2 = markdown_irt_df.shape

# Print the number of rows and columns
print(f"The markdown dataset has {rows2} rows and {columns2} columns.")

The markdown dataset has 98550 rows and 13 columns.


In [4]:
rows2, columns2 = yaml_irt_df.shape

# Print the number of rows and columns
print(f"The yaml dataset has {rows2} rows and {columns2} columns.")

The yaml dataset has 23831 rows and 4 columns.


# Merging the data

In [10]:
# Check column names before merging
print("Markdown IRT columns:", markdown_irt_df.columns)
print("YAML IRT columns:", yaml_irt_df.columns)

# Merge Markdown and YAML IRT characteristics based on 'full_name'
irt_combined_df = markdown_irt_df.merge(yaml_irt_df, on='full_name', how='outer')

# Merge the combined IRT characteristics (Markdown + YAML) with repository characteristics
combined_df = repo_df.merge(irt_combined_df, on='full_name', how='left')

# Debugging: Print available columns after merging
print("Combined dataset columns:", combined_df.columns)

print(f"After merging, the combined dataset has {combined_df.shape[0]} rows and {combined_df.shape[1]} columns.")
print(combined_df.head())

Markdown IRT columns: Index(['name', 'about', 'title', 'labels', 'assignees', 'body', 'IRT_name',
       'full_name', 'has_initial_table', 'IRT_raw', 'IRT_full_name',
       'headlines', 'body_anonymized'],
      dtype='object')
YAML IRT columns: Index(['IRT_name', 'full_name', 'IRT_full_name', 'IRT_raw'], dtype='object')
Combined dataset columns: Index(['full_name', 'is_fork', 'has_issues', 'created_at', 'last_modified',
       'pushed_at', 'main_language', 'total_issues_count', 'open_issues_count',
       'closed_issues_count', 'total_pull_requests_count',
       'open_pull_requests_count', 'closed_pull_requests_count', 'size',
       'topics', 'stargazers_count', 'subscribers_count', 'forks_count',
       'commits_count', 'assignees_count', 'branches_count', 'releases_count',
       'is_archive', 'has_wiki', 'contributors_count', 'open_issues_countv2',
       'closed_issues_countv2', 'total_issues_countv2', 'has_IRT', 'name',
       'about', 'title', 'labels', 'assignees', 'body', '

In [11]:
# Save the combined DataFrame as a CSV file
combined_df.to_csv('combined_girt_data.csv', index=False)

print("Combined data saved as 'combined_girt_data.csv'.")

Combined data saved as 'combined_girt_data.csv'.


In [12]:
rows, columns = combined_df.shape

print(f"The combined dataset has {rows} rows and {columns} columns.")


The combined dataset has 1151463 rows and 44 columns.


# Step 3: Feature Engineering

We’ll now create features that will help CodeBERT prioritize bug issues. The features will be derived from repository characteristics and IRT characteristics. Here are features we will engineer:

### Repository-level Features:

Popularity indicators: number of stars, forks, commits, etc.
Issue activity: total open/closed issues.


### IRT-level Features:

Presence of Markdown or YAML templates.
Length of the issue report template (if available).
Number of headlines or structured fields in the templates.

## Step 3.1: Creating Features from Repository Characteristics

In [13]:
# Feature 1: Presence of IRT (either Markdown or YAML) with safe column access
combined_df['has_irt'] = combined_df.get('IRT_name_markdown', pd.Series()).notnull() | \
                         combined_df.get('IRT_name_yaml', pd.Series()).notnull()

# Feature 2: Repository popularity based on stars
combined_df['is_popular'] = combined_df['stargazers_count'].apply(lambda x: 1 if x > 100 else 0)

# Feature 3: Issue activity (open + closed issues)
combined_df['issue_activity'] = combined_df.get('open_issues_countv2', pd.Series(0)) + \
                                combined_df.get('closed_issues_countv2', pd.Series(0))

# Feature 4: Recent activity (last 6 months)
combined_df['recently_updated'] = combined_df.get('pushed_at', pd.Series()).apply(
    lambda x: 1 if pd.notnull(x) and pd.to_datetime(x) > pd.Timestamp.now() - pd.DateOffset(months=6) else 0
)

# Feature 5: Repository age in days
combined_df['repo_age_days'] = (pd.Timestamp.now() - pd.to_datetime(combined_df.get('created_at', pd.Series()))).dt.days

# Feature 6: Commit frequency (last 3 months)
combined_df['commit_frequency_last_3m'] = combined_df.get('commits_last_3_months', pd.Series(0)) / 90

# Feature 7: Issue resolution time (average in days)
combined_df['avg_issue_resolution_time'] = combined_df.get('issue_resolution_time', pd.Series()).fillna(combined_df.get('issue_resolution_time', pd.Series()).median())

# Feature 8: Presence of README file
combined_df['has_readme'] = combined_df.get('readme_content', pd.Series()).notnull()

# Feature 9: Sentiment Analysis on Issue Titles
combined_df['issue_sentiment'] = combined_df.get('issue_titles', pd.Series()).apply(lambda x: TextBlob(str(x)).sentiment.polarity if pd.notnull(x) else 0)

# Feature 10: Total IRT length (Markdown + YAML)
combined_df['total_irt_length'] = combined_df.get('IRT_raw_markdown', pd.Series('')).apply(lambda x: len(str(x))) + \
                                  combined_df.get('IRT_raw_yaml', pd.Series('')).apply(lambda x: len(str(x)))

# Inspect the new feature set
print(combined_df.head())


C:\Users\Samee\AppData\Local\Temp\ipykernel_20524\1115425492.py:2: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  combined_df['has_irt'] = combined_df.get('IRT_name_markdown', pd.Series()).notnull() | \
C:\Users\Samee\AppData\Local\Temp\ipykernel_20524\1115425492.py:3: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  combined_df.get('IRT_name_yaml', pd.Series()).notnull()
C:\Users\Samee\AppData\Local\Temp\ipykernel_20524\1115425492.py:13: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  combined_df['recently_updated'] = combined_df.get('pushed_at', pd.Series()).apply(
C:\Users\Samee\AppData\Local\Temp\ipykernel_20524\1115425492.py:18: FutureWarning

                          full_name  is_fork  has_issues           created_at  \
0        josch/cycles_johnson_meyer    False        True  2012-07-04 12:02:27   
1   hantsy/angularjs-cakephp-sample    False        True  2013-11-06 12:56:34   
2          metabase/metabase-deploy    False       False  2015-10-08 00:01:18   
3  sakshamsharma/http-over-protocol    False        True  2016-08-30 10:53:30   
4                  sradley/overflow    False        True  2020-10-27 06:30:05   

                   last_modified            pushed_at main_language  \
0  Sun, 01 Jan 2023 16:35:16 GMT  2018-07-05 03:54:37          Java   
1  Wed, 07 Dec 2022 18:29:56 GMT  2015-10-08 05:00:41           PHP   
2  Sat, 10 Dec 2022 12:07:00 GMT  2022-08-05 22:57:15         Shell   
3  Tue, 15 Nov 2022 20:41:40 GMT  2016-09-25 04:59:55           C++   
4  Thu, 01 Dec 2022 11:13:38 GMT  2022-12-18 03:42:19            Go   

   total_issues_count  open_issues_count  closed_issues_count  ...  has_irt  \
0      

In [16]:
pip install textblob

  Obtaining dependency information for textblob from https://files.pythonhosted.org/packages/1e/d6/40aa5aead775582ea0cf35870e5a3f16fab4b967f1ad2debe675f673f923/textblob-0.19.0-py3-none-any.whl.metadata
  Obtaining dependency information for nltk>=3.9 from https://files.pythonhosted.org/packages/4d/66/7d9e26593edda06e8cb531874633f7c2372279c3b0f46235539fe546df8b/nltk-3.9.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/624.3 kB ? eta -:--:--
   ------- -------------------------------- 122.9/624.3 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------  614.4/624.3 kB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 624.3/624.3 kB 5.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------- ----------------------- 0.6/1.5 MB 13.1 MB/s eta 0:00:01
   ------------------------------ --------- 1.2/1.5 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------  1.5/1.5 MB 13.4 MB/s e

In [24]:
import pandas as pd
import torch
from textblob import TextBlob
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from imblearn.over_sampling import SMOTE
from torch.utils.data import Dataset

# Debugging: Print available columns after merging
print("Combined dataset columns:", combined_df.columns)

Combined dataset columns: Index(['full_name', 'is_fork', 'has_issues', 'created_at', 'last_modified',
       'pushed_at', 'main_language', 'total_issues_count', 'open_issues_count',
       'closed_issues_count', 'total_pull_requests_count',
       'open_pull_requests_count', 'closed_pull_requests_count', 'size',
       'topics', 'stargazers_count', 'subscribers_count', 'forks_count',
       'commits_count', 'assignees_count', 'branches_count', 'releases_count',
       'is_archive', 'has_wiki', 'contributors_count', 'open_issues_countv2',
       'closed_issues_countv2', 'total_issues_countv2', 'has_IRT', 'name',
       'about', 'title', 'labels', 'assignees', 'body', 'IRT_name_x',
       'has_initial_table', 'IRT_raw_x', 'IRT_full_name_x', 'headlines',
       'body_anonymized', 'IRT_name_y', 'IRT_full_name_y', 'IRT_raw_y',
       'has_irt', 'is_popular', 'issue_activity', 'recently_updated',
       'repo_age_days', 'commit_frequency_last_3m',
       'avg_issue_resolution_time', 'has_rea

In [25]:
# Step 4.0: Filter for Java repositories
combined_df = combined_df[combined_df['main_language'] == 'Java']

# Initialize CodeBERT tokenizer
tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')

# Use a pipeline as a high-level helper for LLaMA
pipe = pipeline("text-generation", model="huggyllama/llama-7b")

# Load LLaMA tokenizer and model
tokenizer_llama = AutoTokenizer.from_pretrained("huggyllama/llama-7b")
model_llama = AutoModelForCausalLM.from_pretrained("huggyllama/llama-7b")

# Step 4.1: Tokenize the textual fields
def tokenize_text(text):
    if pd.notnull(text):
        return tokenizer(text, padding='max_length', truncation=True, max_length=256, return_tensors="pt")["input_ids"].squeeze()
    return torch.zeros(256, dtype=torch.long)

combined_df['tokenized_body'] = combined_df.apply(
    lambda row: tokenize_text(row.get('IRT_raw_markdown', '') if pd.notnull(row.get('IRT_raw_markdown', '')) 
                              else row.get('IRT_raw_yaml', '')), axis=1)

# Step 4.2.1: Assign priority labels
combined_df['priority_label'] = combined_df['issue_activity'].apply(lambda x: 1 if x > combined_df['issue_activity'].median() else 0)

# Step 4.2.2: Handle missing values
df_cleaned = combined_df.fillna(0)

# Normalize numerical features
scaler = MinMaxScaler()
numerical_features = ['repo_age_days', 'commit_frequency_last_3m', 'avg_issue_resolution_time', 'issue_activity', 'total_irt_length']
df_cleaned[numerical_features] = scaler.fit_transform(df_cleaned[numerical_features])

# Encode categorical features if necessary
if 'priority_label' in df_cleaned.columns:
    label_encoder = LabelEncoder()
    df_cleaned['priority_label'] = label_encoder.fit_transform(df_cleaned['priority_label'])

# Step 4.2.3: Apply SMOTE for class balancing
smote = SMOTE(random_state=42)
numeric_features_only = df_cleaned.select_dtypes(include=['number']).drop(columns=['priority_label'])
x_resampled, y_resampled = smote.fit_resample(numeric_features_only, df_cleaned['priority_label'])

# Create new balanced DataFrame
resampled_df = pd.DataFrame(x_resampled, columns=numeric_features_only.columns)
resampled_df['priority_label'] = y_resampled

# Step 4.2.3: Split into training and validation sets
train_df, test_df = train_test_split(resampled_df, test_size=0.2, random_state=42)

# Define PyTorch Dataset class
class HybridBugReportDataset(Dataset):
    def __init__(self, dataframe):
        self.tokenized_texts = torch.stack(list(dataframe['tokenized_body']))
        self.labels = torch.tensor(dataframe['priority_label'].values, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.tokenized_texts[idx], self.labels[idx]

train_dataset = HybridBugReportDataset(train_df)
test_dataset = HybridBugReportDataset(test_df)

# Load CodeBERT model
codebert_model = RobertaForSequenceClassification.from_pretrained('microsoft/codebert-base', num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

# Trainer setup for CodeBERT
codebert_trainer = Trainer(
    model=codebert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Step 5: Fine-Tuning CodeBERT
codebert_trainer.train()

# Save the fine-tuned model
codebert_model.save_pretrained("./fine_tuned_codebert")
tokenizer.save_pretrained("./fine_tuned_codebert")

print("Hybrid model integration complete!")


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

C:\Users\Samee\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Samee\.cache\huggingface\hub\models--huggyllama--llama-7b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

KeyError: 'tokenized_body'